# 02 — WHO GHO Cleaning

- Source: WHO GHO, downloaded [date]
- Indicators: NCD_BMI_30A, NCD_DIABETES_PREVALENCE_AGESTD, NCDMORT3070
- Filters: country rows only; both sexes; age-standardised; obesity and diabetes at 18+; mortality at 30 to 69
- Dropped: CRUDE diabetes, sex-specific rows, diabetes 30+ rows
- Coverage: obesity 1980 to 2024 (199 countries), diabetes 1990 to 2022 (199), mortality 2000 to 2023 (185)
- Missing values: left as NaN, not imputed
- Low/high columns: uncertainty bounds from WHO, kept for display in Tableau

What to take note of
- ISO codes must match GDD. Once GDD is cleaned, check the mismatches in both directions (the code from my earlier message). Typical culprits are territories and Kosovo.
- Don’t impute. Gaps are real, and Tableau can show them.
- Overlap window is 2000 to 2022 for all three indicators. Mortality can only be paired with a diet year that is at least 10 years earlier than 2000, so check GDD’s time points.
- Obesity is the headline outcome, with diabetes and mortality secondary.
- Obesity runs to 2024 and some values are modelled projections, so don’t over-interpret the latest years.
- Commit the notebook with outputs cleared, and share the cleaned CSV through your team drive if data/cleaned is gitignored.

Step 1. Load with helper function    

In [24]:
import pandas as pd
import io

def load_gho(path):
    with open(path, encoding="utf-8-sig", newline="") as f:
        text = f.read()

    # Find where the real header starts: a line that begins with IndicatorCode,
    # not just the word appearing inside the quoted preamble
    pos = None
    if text.startswith("IndicatorCode"):
        pos = 0
    else:
        idx = text.find("\nIndicatorCode,")
        if idx != -1:
            pos = idx + 1
    if pos is None:
        raise ValueError(f"No header line starting with IndicatorCode in {path}")

    d = pd.read_csv(io.StringIO(text[pos:]))
    d.columns = d.columns.str.strip()
    return d

mort = load_gho(RAW / "Cardiovascular diseases, cancer, diabetes, or chronic respiratory disease mortality WHO.csv")
obes = load_gho(RAW / "Prevalence of obesity among adults WHO.csv")
diab = load_gho(RAW / "Prevalence of diabetes WHO.csv")

for name, d in [("mort", mort), ("obes", obes), ("diab", diab)]:
    print(name, d.shape, "| has IndicatorCode:", "IndicatorCode" in d.columns)
    print(list(d.columns)[:6])

mort (13320, 34) | has IndicatorCode: True
['IndicatorCode', 'Indicator', 'ValueType', 'ParentLocationCode', 'ParentLocation', 'Location type']
obes (26865, 34) | has IndicatorCode: True
['IndicatorCode', 'Indicator', 'ValueType', 'ParentLocationCode', 'ParentLocation', 'Location type']
diab (78804, 34) | has IndicatorCode: True
['IndicatorCode', 'Indicator', 'ValueType', 'ParentLocationCode', 'ParentLocation', 'Location type']


Step 2: Inspect before filtering

In [25]:
for name, d in [("mort", mort), ("obes", obes), ("diab", diab)]:
    print("=====", name, d.shape)
    print(d["IndicatorCode"].value_counts())
    print(d["Location type"].value_counts())
    print(d[["Dim1ValueCode", "Dim2ValueCode"]].drop_duplicates())
    print("years:", d["Period"].min(), "to", d["Period"].max(),
          "| countries:", d["SpatialDimValueCode"].nunique())

===== mort (13320, 34)
IndicatorCode
NCDMORT3070    13320
Name: count, dtype: int64
Location type
Country    13320
Name: count, dtype: int64
  Dim1ValueCode        Dim2ValueCode
0      SEX_FMLE  AGEGROUP_YEARS30-69
1       SEX_MLE  AGEGROUP_YEARS30-69
2      SEX_BTSX  AGEGROUP_YEARS30-69
years: 2000 to 2023 | countries: 185
===== obes (26865, 34)
IndicatorCode
NCD_BMI_30A    26865
Name: count, dtype: int64
Location type
Country    26865
Name: count, dtype: int64
  Dim1ValueCode          Dim2ValueCode
0      SEX_BTSX  AGEGROUP_YEARS18-PLUS
1       SEX_MLE  AGEGROUP_YEARS18-PLUS
2      SEX_FMLE  AGEGROUP_YEARS18-PLUS
years: 1980 to 2024 | countries: 199
===== diab (78804, 34)
IndicatorCode
NCD_DIABETES_PREVALENCE_AGESTD    39402
NCD_DIABETES_PREVALENCE_CRUDE     39402
Name: count, dtype: int64
Location type
Country    78804
Name: count, dtype: int64
  Dim1ValueCode          Dim2ValueCode
0      SEX_BTSX  AGEGROUP_YEARS30-PLUS
1      SEX_BTSX  AGEGROUP_YEARS18-PLUS
2       SEX_MLE  AGEGRO

Step 3: Cleaning

In [26]:
def clean_gho(d, indicator_code, out_name, agegroup, sex="SEX_BTSX"):
    d = d[(d["IndicatorCode"] == indicator_code)
          & (d["Location type"] == "Country")
          & (d["Dim1ValueCode"] == sex)
          & (d["Dim2ValueCode"] == agegroup)]
    d = d.dropna(subset=["SpatialDimValueCode", "FactValueNumeric"])

    d = d.rename(columns={
        "SpatialDimValueCode": "iso3",
        "Location": "country_name",
        "ParentLocation": "who_region",
        "Period": "year",
        "FactValueNumeric": out_name,
        "FactValueNumericLow": f"{out_name}_low",
        "FactValueNumericHigh": f"{out_name}_high",
    })
    d["iso3"] = d["iso3"].str.upper().str.strip()
    d["year"] = d["year"].astype(int)
    return d[["iso3", "country_name", "who_region", "year",
              out_name, f"{out_name}_low", f"{out_name}_high"]]

obes_c = clean_gho(obes, "NCD_BMI_30A", "obesity_pct", "AGEGROUP_YEARS18-PLUS")
diab_c = clean_gho(diab, "NCD_DIABETES_PREVALENCE_AGESTD", "diabetes_pct", "AGEGROUP_YEARS18-PLUS")
mort_c = clean_gho(mort, "NCDMORT3070", "premature_ncd_mort_pct", "AGEGROUP_YEARS30-69")

Step 4. Validate expected results

In [27]:
for name, d, col in [("obes", obes_c, "obesity_pct"),
                     ("diab", diab_c, "diabetes_pct"),
                     ("mort", mort_c, "premature_ncd_mort_pct")]:
    print(name, d.shape)
    print("  duplicate iso3-year rows:", d.duplicated(["iso3", "year"]).sum())   # must be 0
    print("  value range:", d[col].min(), d[col].max())
    print("  years:", d["year"].min(), "-", d["year"].max(), "| countries:", d["iso3"].nunique())

obes (8955, 7)
  duplicate iso3-year rows: 0
  value range: 0.14 75.87
  years: 1980 - 2024 | countries: 199
diab (6567, 7)
  duplicate iso3-year rows: 0
  value range: 2.33 34.82
  years: 1990 - 2022 | countries: 199
mort (4440, 7)
  duplicate iso3-year rows: 0
  value range: 5.5 45.4
  years: 2000 - 2023 | countries: 185


Step 5. Merge 3 tables

In [28]:
# Outer join so no country-year is lost just because one indicator is missing
who = (obes_c
       .merge(diab_c.drop(columns=["country_name", "who_region"]), on=["iso3", "year"], how="outer")
       .merge(mort_c.drop(columns=["country_name", "who_region"]), on=["iso3", "year"], how="outer"))

# The outer join leaves country_name / who_region empty for rows that only exist in diab or mort,
# so rebuild them from one lookup
lookup = (pd.concat([obes_c, diab_c, mort_c])[["iso3", "country_name", "who_region"]]
          .dropna().drop_duplicates("iso3"))
who = who.drop(columns=["country_name", "who_region"]).merge(lookup, on="iso3", how="left")

# Column order and sort
cols = ["iso3", "country_name", "who_region", "year",
        "obesity_pct", "obesity_pct_low", "obesity_pct_high",
        "diabetes_pct", "diabetes_pct_low", "diabetes_pct_high",
        "premature_ncd_mort_pct", "premature_ncd_mort_pct_low", "premature_ncd_mort_pct_high"]
who = who[cols].sort_values(["iso3", "year"]).reset_index(drop=True)

Step 6. Validate merged tables

In [29]:
assert who.duplicated(["iso3", "year"]).sum() == 0, "duplicate country-year rows"
assert who["country_name"].notna().all(), "some iso3 codes have no country name"

print("shape:", who.shape)
print("countries:", who["iso3"].nunique(), "| years:", who["year"].min(), "-", who["year"].max())
print("\nmissing share per column:")
print(who.isna().mean().round(3))

# Rows where all three indicators exist
full = who.dropna(subset=["obesity_pct", "diabetes_pct", "premature_ncd_mort_pct"])
print("\nrows with all three indicators:", len(full), "| year range:", full["year"].min(), "-", full["year"].max())

# Countries that lack a given indicator entirely
for c in ["obesity_pct", "diabetes_pct", "premature_ncd_mort_pct"]:
    n = who.groupby("iso3")[c].apply(lambda s: s.notna().any())
    print(f"{c}: {(~n).sum()} countries have no data at all")

shape: (8955, 13)
countries: 199 | years: 1980 - 2024

missing share per column:
iso3                           0.000
country_name                   0.000
who_region                     0.000
year                           0.000
obesity_pct                    0.000
obesity_pct_low                0.000
obesity_pct_high               0.000
diabetes_pct                   0.267
diabetes_pct_low               0.267
diabetes_pct_high              0.267
premature_ncd_mort_pct         0.504
premature_ncd_mort_pct_low     0.504
premature_ncd_mort_pct_high    0.504
dtype: float64

rows with all three indicators: 4255 | year range: 2000 - 2022
obesity_pct: 0 countries have no data at all
diabetes_pct: 0 countries have no data at all
premature_ncd_mort_pct: 14 countries have no data at all


Step 7. Save and document

In [30]:
who.to_csv(CLEAN / "who_gho_clean.csv", index=False)

Schema check

In [33]:
import pandas as pd

def check_schema(df, name, grain=("iso3", "year")):
    problems = []
    for k in grain:
        if k not in df.columns:
            problems.append(f"missing key column: {k}")
    if not problems:
        if df.duplicated(list(grain)).any():
            problems.append(f"{df.duplicated(list(grain)).sum()} duplicate rows on {grain}")
        if not df["iso3"].astype(str).str.fullmatch(r"[A-Z]{3}").all():
            problems.append("iso3 not all uppercase 3-letter codes")
        if "year" in grain and df["year"].dtype.kind != "i":
            problems.append(f"year dtype is {df['year'].dtype}, expected int")
    bad_names = [c for c in df.columns if c != c.lower() or " " in c]
    if bad_names:
        problems.append(f"non snake_case columns: {bad_names}")
    zeros = [c for c in df.select_dtypes("number").columns if (df[c] == -999).any()]
    if zeros:
        problems.append(f"-999 sentinel values in: {zeros}")

    print(f"{name}: {df.shape} | iso3: {df['iso3'].nunique()} | "
          f"years: {df['year'].min()}-{df['year'].max()}")
    print("  OK" if not problems else "  PROBLEMS:\n   - " + "\n   - ".join(problems))
    return not problems


check_schema(who, "who_gho")

who_gho: (8955, 13) | iso3: 199 | years: 1980-2024
  OK


True